# 🏭 Deep Research EDA: Macro Market & Shop Fulfillment
This analysis isolates exactly *why* certain random shop spawns result in 190k ceilings while others top out at 120k. By tracking market inventory and prices over time, we can visualize the specific crop demand profiles.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (16, 8)

df = pd.read_csv('market_dynamics.csv')



## 1. Price Trajectories: Super Elite (180k+) vs Standard Elite (120k)
The 180k replays have shops that naturally consume exactly what the Grandmaster produces, keeping the price artificially high despite massive supply dumping.


In [ ]:
super_elite = df[df['Final_Score'] >= 180000].groupby('Day')[['Price_STRAWBERRY', 'Price_MELON', 'Price_MILK', 'Price_WOOL']].mean().reset_index()
standard_elite = df[(df['Final_Score'] >= 120000) & (df['Final_Score'] < 130000)].groupby('Day')[['Price_STRAWBERRY', 'Price_MELON', 'Price_MILK', 'Price_WOOL']].mean().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].plot(super_elite['Day'], super_elite['Price_STRAWBERRY'], label='180k+ Thread (Ice Cream/Smoothie)', color='red', linewidth=3)
axes[0, 0].plot(standard_elite['Day'], standard_elite['Price_STRAWBERRY'], label='120k Thread', color='salmon', linestyle='--')
axes[0, 0].set_title('Strawberry Price Decay')
axes[0, 0].legend()

axes[0, 1].plot(super_elite['Day'], super_elite['Price_MELON'], label='180k+ Thread', color='green', linewidth=3)
axes[0, 1].plot(standard_elite['Day'], standard_elite['Price_MELON'], label='120k Thread', color='lightgreen', linestyle='--')
axes[0, 1].set_title('Melon Price Decay')
axes[0, 1].legend()

axes[1, 0].plot(super_elite['Day'], super_elite['Price_MILK'], label='180k+ Thread', color='blue', linewidth=3)
axes[1, 0].plot(standard_elite['Day'], standard_elite['Price_MILK'], label='120k Thread', color='lightblue', linestyle='--')
axes[1, 0].set_title('Milk Price Decay')
axes[1, 0].legend()

axes[1, 1].plot(super_elite['Day'], super_elite['Price_WOOL'], label='180k+ Thread', color='gray', linewidth=3)
axes[1, 1].plot(standard_elite['Day'], standard_elite['Price_WOOL'], label='120k Thread', color='lightgray', linestyle='--')
axes[1, 1].set_title('Wool Price Decay')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 2. Inventory Accumulation (Shop Fulfillment Rates)
When the market inventory spikes above 10,000, it means the players are producing faster than the shops can consume. Let's look at the final Day 30 market inventory delta for both tiers.


In [ ]:
day30_df = df[df['Day'] == 30]
day30_se = day30_df[day30_df['Final_Score'] >= 180000].mean(numeric_only=True)
day30_std = day30_df[(day30_df['Final_Score'] >= 120000) & (day30_df['Final_Score'] < 130000)].mean(numeric_only=True)

inv_cols = ['Inv_WHEAT', 'Inv_CARROT', 'Inv_STRAWBERRY', 'Inv_MELON', 'Inv_MILK', 'Inv_WOOL']
labels = ['WHEAT', 'CARROT', 'STRAWBERRY', 'MELON', 'MILK', 'WOOL']

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, [day30_se[c] - 10000 for c in inv_cols], width, label='180k+ Sinks', color='purple')
rects2 = ax.bar(x + width/2, [day30_std[c] - 10000 for c in inv_cols], width, label='120k Sinks', color='orange')

ax.set_ylabel('Market Surplus (Inventory > 10k)')
ax.set_title('Final Day 30 Market Glut (Higher = Price Crash)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.axhline(0, color='black', linewidth=1)
plt.show()
